In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

lakehouse = "lh_bronze_usaspending"
agencies = ["SBA", "EPA", "DOL", "VA"]
award_types = ["assistance", "contracts"]

merge_keys = [
    "award_unique_key",
    "parent_award_id_piid",
    "federal_account_symbol",
    "program_activity_code",
    "program_activity_name",
    "object_class_code",
    "direct_or_reimbursable_funding_source",
    "disaster_emergency_fund_code",
    "submission_period",
    "program_activity_reporting_key",
]

# Only these columns are true numeric measures. Everything else (ids, codes,
# account symbols, dates-as-text) is kept as string, because inferSchema can
# guess different types per agency file and break later merges — e.g. a
# program_activity_code that looks numeric in one file ('12345')
# and alphanumeric in another ('68HF05').
numeric_cols = [
    "transaction_obligated_amount",
    "gross_outlay_amount_FYB_to_period_end",
    "USSGL487200_downward_adj_prior_year_prepaid_undeliv_order_oblig",
    "USSGL497200_downward_adj_of_prior_year_paid_deliv_orders_oblig",
]

StatementMeta(, 1b6caf90-8539-48e1-bec7-78dbafb5197f, 3, Finished, Available, Finished, False)

In [2]:
def list_fiscal_years(award_type: str, agency: str):
    """Auto-detect which FY folders exist for this agency/award type,
    so new years (e.g. FY2026) are picked up without changing the code."""
    base_path = f"Files/raw/award/{award_type}/{agency}"
    try:
        items = mssparkutils.fs.ls(base_path)
    except Exception:
        return []
    return [i.name for i in items if i.isDir and i.name.startswith("FY")]

StatementMeta(, 1b6caf90-8539-48e1-bec7-78dbafb5197f, 4, Finished, Available, Finished, False)

In [3]:
def load_incremental(award_type: str, agency: str, fiscal_year: str):
    path = f"Files/raw/award/{award_type}/{agency}/{fiscal_year}"
    table_name = f"{lakehouse}.dbo.bronze_{award_type}"

    df = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "false")
          .option("quote", '"')
          .option("escape", '"')       # handles doubled quotes inside quoted fields, e.g. "" 
          .option("multiLine", "true") # handles embedded newlines inside quoted fields
          #.option("mode", "PERMISSIVE")
          #.option("columnNameOfCorruptRecord", "_corrupt_record")
          .load(path)
          .withColumn("agency_code", F.lit(agency))
          .withColumn("fiscal_year_folder", F.lit(fiscal_year)))

    for c in numeric_cols:
        df = df.withColumn(c, F.col(c).cast("double"))

    # NEW: timestamp when this row entered Bronze — this is what Silver
    # will use to know what's new since its last run.
    df = df.withColumn("_bronze_load_ts", F.current_timestamp())
    
    if not spark.catalog.tableExists(table_name):
        # Very first load of this table overall (any agency).
        print(f"Creating {table_name} (first load, {agency}/{fiscal_year})")
        df.write.format("delta").mode("overwrite").saveAsTable(table_name)
        return

    # Check whether THIS agency + fiscal year already has rows in the table —
    # not whether the table itself exists. A new agency or new year should be
    # a cheap append, not a full-table merge scan.
    already_loaded = spark.sql(f"""
        SELECT COUNT(*) AS cnt FROM {table_name}
        WHERE agency_code = '{agency}' AND fiscal_year_folder = '{fiscal_year}'
    """).collect()[0]["cnt"]

    if already_loaded == 0:
        print(f"Appending {agency}/{fiscal_year} (new combination, no merge needed)")
        df.write.format("delta").mode("append").saveAsTable(table_name)
        return

    # This agency + year has been loaded before (e.g. re-running FY2025 after
    # a corrected file, or reloading FY2026 a second time) — now a real merge
    # is needed to catch updated rows via last_modified_date.
    print(f"Merging into {table_name} for agency {agency}, year {fiscal_year} (existing data found)")
    bronze = DeltaTable.forName(spark, table_name)
    join_condition = " AND ".join(f"t.{k} <=> s.{k}" for k in merge_keys)

    (bronze.alias("t")
        .merge(df.alias("s"), join_condition)
        .whenMatchedUpdateAll(condition="s.last_modified_date > t.last_modified_date")
        .whenNotMatchedInsertAll()
        .execute())

StatementMeta(, 1b6caf90-8539-48e1-bec7-78dbafb5197f, 5, Finished, Available, Finished, False)

In [4]:
for award_type in award_types:
    for agency in agencies:
        for fiscal_year in list_fiscal_years(award_type, agency):
            try:
                load_incremental(award_type, agency, fiscal_year)
            except Exception as e:
                print(f"Skipped {award_type}/{agency}/{fiscal_year}: {e}")

StatementMeta(, 1b6caf90-8539-48e1-bec7-78dbafb5197f, 6, Finished, Available, Finished, False)

Creating lh_bronze_usaspending.dbo.bronze_assistance (first load, SBA/FY2025)
Appending EPA/FY2025 (new combination, no merge needed)
Appending DOL/FY2025 (new combination, no merge needed)
Appending VA/FY2025 (new combination, no merge needed)
Creating lh_bronze_usaspending.dbo.bronze_contracts (first load, SBA/FY2025)
Appending EPA/FY2025 (new combination, no merge needed)
Appending DOL/FY2025 (new combination, no merge needed)
Appending VA/FY2025 (new combination, no merge needed)
